# Week 8 — Module 19
Function-specific optimisation and surrogate validation.

## Load observations

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Use the Week 7 datasets as the starting point for Week 8.
search_roots = [Path.cwd(), *Path.cwd().parents]
relative_locations = [
    Path("7. Module 18/updated_data_week_7"),
    Path("updated_data_week_7"),
    Path("8. Module 19/Data"),
    Path("Data"),
]
data_folder = next(
    (base / location for base in search_roots for location in relative_locations
     if all((base / location / f"function_{i}.csv").is_file() for i in range(1, 9))),
    None,
)
if data_folder is None:
    raise FileNotFoundError("Could not find all eight Week 7 function CSVs. Place them in updated_data_week_7 or Data.")

data = {i: pd.read_csv(data_folder / f"function_{i}.csv") for i in range(1, 9)}
print(f"Loaded eight datasets from {data_folder}")

Loaded eight datasets from C:\Users\charl\OneDrive - thesuperyachtgroup.com\Desktop\Imperial AI course\7. Capstone Project\8. Module 19\Data


## Week 7 observations

In [2]:
# Week 7 submitted points
new_points = {
    1: [0.858598, 0.697368],
    2: [0.706079, 0.158030],
    3: [0.450320, 0.561567, 0.490526],
    4: [0.384332, 0.391111, 0.336887, 0.416488],
    5: [1.000000, 1.000000, 1.000000, 1.000000],
    6: [0.387946, 0.435172, 0.345126, 0.743513, 0.076825],
    7: [0.054205, 0.216830, 0.421532, 0.346756, 0.385001, 0.543902],
    8: [0.030327, 0.231267, 0.050266, 0.208295,
        0.818520, 0.414744, 0.307096, 0.583908]
}

# Returned outputs
new_outputs = {
    1: 1.539769610658716e-40,
    2: 0.5699038779230302,
    3: -0.010189939911026829,
    4: 0.18691172535641742,
    5: 8662.4825,
    6: -0.3886868054486413,
    7: 2.369288578893768,
    8: 9.9308110991456
}

In [3]:
week7_results = []
for i in range(1, 9):
    previous_best = data[i]["output"].max()
    new_output = new_outputs[i]
    week7_results.append({"Function": i, "Previous best": previous_best,
                          "New output": new_output, "Improved": new_output > previous_best})
print(pd.DataFrame(week7_results).round(6).to_string(index=False))

 Function  Previous best  New output  Improved
        1       0.000000    0.000000     False
        2       0.746962    0.569904     False
        3      -0.010190   -0.010190     False
        4       0.186912    0.186912      True
        5    8662.482500 8662.482500     False
        6      -0.379740   -0.388687     False
        7       2.929370    2.369289     False
        8       9.930811    9.930811     False


In [4]:
for i in range(1, 9):
    input_columns = [col for col in data[i].columns if col != "output"]
    point = np.asarray(new_points[i], dtype=float)
    existing = data[i][input_columns].to_numpy(dtype=float)
    matching = np.flatnonzero(np.all(np.isclose(existing, point, atol=5e-7, rtol=0), axis=1))
    if len(matching):
        recorded = float(data[i].iloc[matching[0]]["output"])
        if not np.isclose(recorded, new_outputs[i], atol=1e-6, rtol=1e-8):
            raise ValueError(f"F{i}: existing Week 7 point has a different output")
    else:
        row = dict(zip(input_columns, point))
        row["output"] = new_outputs[i]
        data[i] = pd.concat([data[i], pd.DataFrame([row])], ignore_index=True)
print("Observations:", ", ".join(f"F{i}: {len(data[i])}" for i in range(1, 9)))

Observations: F1: 17, F2: 17, F3: 22, F4: 37, F5: 27, F6: 27, F7: 37, F8: 47


## Gaussian Process models

In [5]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
gp_models = {}
for i in range(1, 9):
    X = data[i].drop(columns='output').values
    y = data[i]['output'].values
    gp = GaussianProcessRegressor(kernel=RBF(length_scale=1.0, length_scale_bounds=(1e-05, 100000.0)), normalize_y=True, n_restarts_optimizer=10, random_state=42)
    gp.fit(X, y)
    gp_models[i] = gp

print("GP length scales:", ", ".join(f"F{i}: {gp_models[i].kernel_.length_scale:.4g}" for i in range(1, 9)))

GP length scales: F1: 1e-05, F2: 1e-05, F3: 0.1995, F4: 0.4159, F5: 0.2731, F6: 0.5809, F7: 0.3021, F8: 1.036


## Prediction diagnostics

In [6]:
rng = np.random.default_rng(42)
candidate_sets = {}
for i in range(1, 9):
    n_dims = data[i].shape[1] - 1
    candidates = rng.random((100000, n_dims))
    candidate_sets[i] = candidates
    pred_mean, pred_std = gp_models[i].predict(candidates, return_std=True)

## Neural-network ensembles

In [7]:
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
nn_models = {}
nn_x_scalers = {}
nn_y_scalers = {}
nn_training_losses = {}
n_ensemble = 5
for i in range(1, 9):
    X = data[i].drop(columns='output').values.astype(float)
    y = data[i]['output'].values.astype(float).reshape(-1, 1)
    x_scaler = StandardScaler()
    y_scaler = StandardScaler()
    X_scaled = x_scaler.fit_transform(X)
    y_scaled = y_scaler.fit_transform(y).ravel()
    models = []
    losses = []
    for seed in range(n_ensemble):
        model = MLPRegressor(hidden_layer_sizes=(20, 20), activation='relu', solver='lbfgs', alpha=0.001, max_iter=5000, random_state=seed)
        model.fit(X_scaled, y_scaled)
        models.append(model)
        losses.append(model.loss_)
    nn_models[i] = models
    nn_x_scalers[i] = x_scaler
    nn_y_scalers[i] = y_scaler
    nn_training_losses[i] = losses

print("NN training losses:", ", ".join(f"F{i}: {np.mean(nn_training_losses[i]):.6f}" for i in range(1, 9)))

NN training losses: F1: 0.000963, F2: 0.010989, F3: 0.000416, F4: 0.000284, F5: 0.000230, F6: 0.000370, F7: 0.000338, F8: 0.000381


## Neural-network LOOCV

In [8]:
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
import numpy as np
nn_loocv_results = {}
for i in range(1, 9):
    X = data[i].drop(columns='output').values.astype(float)
    y = data[i]['output'].values.astype(float)
    nn_predictions = []
    baseline_predictions = []
    actual_values = []
    for j in range(len(X)):
        mask = np.ones(len(X), dtype=bool)
        mask[j] = False
        X_train = X[mask]
        y_train = y[mask]
        X_test = X[j:j + 1]
        y_test = y[j]
        x_scaler = StandardScaler()
        y_scaler = StandardScaler()
        X_train_scaled = x_scaler.fit_transform(X_train)
        X_test_scaled = x_scaler.transform(X_test)
        y_train_scaled = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()
        model = MLPRegressor(hidden_layer_sizes=(20, 20), activation='relu', solver='lbfgs', alpha=0.001, max_iter=5000, random_state=42)
        model.fit(X_train_scaled, y_train_scaled)
        pred_scaled = model.predict(X_test_scaled)[0]
        pred = y_scaler.inverse_transform([[pred_scaled]])[0, 0]
        nn_predictions.append(pred)
        baseline_predictions.append(np.mean(y_train))
        actual_values.append(y_test)
    nn_mae = mean_absolute_error(actual_values, nn_predictions)
    baseline_mae = mean_absolute_error(actual_values, baseline_predictions)
    beats_baseline = nn_mae < baseline_mae
    nn_loocv_results[i] = {'nn_mae': nn_mae, 'baseline_mae': baseline_mae, 'beats_baseline': beats_baseline}

print(pd.DataFrame([{"Function": i, **nn_loocv_results[i]} for i in range(1, 9)]).round(6).to_string(index=False))

 Function     nn_mae  baseline_mae  beats_baseline
        1   0.001704      0.001050           False
        2   0.219676      0.237993            True
        3   0.061985      0.051662           False
        4   1.825570      7.284354            True
        5 230.163312   2049.255132            True
        6   0.241315      0.513894            True
        7   0.399610      0.639930            True
        8   0.354980      0.962395            True


## GP acquisition candidates

In [9]:
from scipy.stats import norm
from scipy.spatial.distance import cdist
import numpy as np
kappa_values = [0.5, 1.5]
xi_percentages = [0.0, 0.05]
gp_candidate_results = {}
for i in range(1, 9):
    X = data[i].drop(columns='output').values.astype(float)
    y = data[i]['output'].values.astype(float)
    candidates = candidate_sets[i]
    mean, std = gp_models[i].predict(candidates, return_std=True)
    current_best = np.max(y)
    y_std = np.std(y)
    results = {'UCB': {}, 'EI': {}}
    for kappa in kappa_values:
        ucb = mean + kappa * std
        idx = np.argmax(ucb)
        point = candidates[idx]
        nearest_distance = np.min(cdist(point.reshape(1, -1), X))
        results['UCB'][kappa] = {'point': point, 'predicted': mean[idx], 'uncertainty': std[idx], 'score': ucb[idx], 'nearest_distance': nearest_distance}
    for xi_pct in xi_percentages:
        xi = xi_pct * y_std
        improvement = mean - current_best - xi
        with np.errstate(divide='ignore', invalid='ignore'):
            Z = np.divide(improvement, std, out=np.zeros_like(improvement), where=std > 0)
        ei = improvement * norm.cdf(Z) + std * norm.pdf(Z)
        ei[std == 0] = 0
        idx = np.argmax(ei)
        point = candidates[idx]
        nearest_distance = np.min(cdist(point.reshape(1, -1), X))
        results['EI'][xi_pct] = {'point': point, 'predicted': mean[idx], 'uncertainty': std[idx], 'score': ei[idx], 'nearest_distance': nearest_distance}
    gp_candidate_results[i] = results

## Neural-network candidates

In [10]:
from scipy.spatial.distance import cdist
nn_candidate_results = {}
for i in range(1, 9):
    X = data[i].drop(columns='output').values.astype(float)
    y = data[i]['output'].values.astype(float)
    candidates = candidate_sets[i]
    candidates_scaled = nn_x_scalers[i].transform(candidates)
    model_predictions = []
    for model in nn_models[i]:
        pred_scaled = model.predict(candidates_scaled)
        pred = nn_y_scalers[i].inverse_transform(pred_scaled.reshape(-1, 1)).ravel()
        model_predictions.append(pred)
    model_predictions = np.array(model_predictions)
    mean_prediction = np.mean(model_predictions, axis=0)
    model_disagreement = np.std(model_predictions, axis=0)
    idx = np.argmax(mean_prediction)
    selected_point = candidates[idx]
    selected_prediction = mean_prediction[idx]
    selected_disagreement = model_disagreement[idx]
    best_idx = np.argmax(y)
    current_best_point = X[best_idx]
    distance_from_best = np.linalg.norm(selected_point - current_best_point)
    nearest_distance = np.min(cdist(selected_point.reshape(1, -1), X))
    nn_candidate_results[i] = {'point': selected_point, 'predicted': selected_prediction, 'disagreement': selected_disagreement, 'distance_from_best': distance_from_best, 'nearest_distance': nearest_distance}

## GP–NN comparison

In [11]:
gp_nn_distances = {}
for i in range(1, 9):
    nn_point = nn_candidate_results[i]['point']
    distances = {'UCB k=0.5': np.linalg.norm(nn_point - gp_candidate_results[i]['UCB'][0.5]['point']), 'UCB k=1.5': np.linalg.norm(nn_point - gp_candidate_results[i]['UCB'][1.5]['point']), 'EI xi=0%': np.linalg.norm(nn_point - gp_candidate_results[i]['EI'][0.0]['point']), 'EI xi=5%': np.linalg.norm(nn_point - gp_candidate_results[i]['EI'][0.05]['point'])}
    gp_nn_distances[i] = distances

## Neighbourhood evidence

In [12]:
neighbourhood_results = {}
for i in range(1, 9):
    X = data[i].drop(columns='output').values.astype(float)
    y = data[i]['output'].values.astype(float)
    candidates_to_check = {'UCB k=0.5': gp_candidate_results[i]['UCB'][0.5]['point'], 'UCB k=1.5': gp_candidate_results[i]['UCB'][1.5]['point'], 'EI xi=0%': gp_candidate_results[i]['EI'][0.0]['point'], 'EI xi=5%': gp_candidate_results[i]['EI'][0.05]['point'], 'NN': nn_candidate_results[i]['point']}
    function_results = {}
    for method, point in candidates_to_check.items():
        distances = cdist(point.reshape(1, -1), X).ravel()
        nearest_idx = np.argsort(distances)[:3]
        nearest_outputs = y[nearest_idx]
        nearest_distances = distances[nearest_idx]
        function_results[method] = {'point': point, 'nearest_indices': nearest_idx, 'nearest_outputs': nearest_outputs, 'nearest_distances': nearest_distances}
    neighbourhood_results[i] = function_results

## Function 1: alternative kernels

In [13]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, WhiteKernel
from sklearn.metrics import mean_absolute_error
import numpy as np
X = data[1].drop(columns='output').values.astype(float)
y = data[1]['output'].values.astype(float)
kernels = {'RBF': RBF(length_scale=0.1, length_scale_bounds=(1e-05, 100.0)), 'Matern 3/2': Matern(length_scale=0.1, length_scale_bounds=(1e-05, 100.0), nu=1.5), 'Matern 5/2': Matern(length_scale=0.1, length_scale_bounds=(1e-05, 100.0), nu=2.5), 'RBF + noise': RBF(length_scale=0.1, length_scale_bounds=(1e-05, 100.0)) + WhiteKernel(noise_level=1e-05, noise_level_bounds=(1e-08, 1.0))}
for name, kernel in kernels.items():
    predictions = []
    actual = []
    baseline_predictions = []
    for j in range(len(X)):
        mask = np.ones(len(X), dtype=bool)
        mask[j] = False
        X_train = X[mask]
        y_train = y[mask]
        X_test = X[j:j + 1]
        y_test = y[j]
        gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=10, random_state=42)
        gp.fit(X_train, y_train)
        pred = gp.predict(X_test)[0]
        predictions.append(pred)
        actual.append(y_test)
        baseline_predictions.append(np.mean(y_train))
    gp_mae = mean_absolute_error(actual, predictions)
    baseline_mae = mean_absolute_error(actual, baseline_predictions)
    final_gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=10, random_state=42)
    final_gp.fit(X, y)

## Function 1: space-filling search

In [14]:
X_f1 = data[1].drop(columns='output').values.astype(float)
rng = np.random.default_rng(42)
f1_exploration_candidates = rng.random((100000, 2))
distances = cdist(f1_exploration_candidates, X_f1)
nearest_distances = np.min(distances, axis=1)
order = np.argsort(nearest_distances)[::-1]
selected = []
for idx in order:
    point = f1_exploration_candidates[idx]
    if len(selected) == 0:
        selected.append(idx)
    else:
        existing_selected = f1_exploration_candidates[selected]
        if np.min(cdist(point.reshape(1, -1), existing_selected)) > 0.1:
            selected.append(idx)
    if len(selected) == 5:
        break
for rank, idx in enumerate(selected, start=1):
    point = f1_exploration_candidates[idx]

## Function 1: candidate comparison

In [15]:
f1_top_candidates = np.array([[0.998294, 0.000709], [0.684654, 0.001334], [0.006886, 0.695109]])
X = data[1].drop(columns='output').values.astype(float)
y = data[1]['output'].values.astype(float)
gp = gp_models[1]
candidates_scaled = nn_x_scalers[1].transform(f1_top_candidates)
nn_preds = []
for model in nn_models[1]:
    pred_scaled = model.predict(candidates_scaled)
    pred = nn_y_scalers[1].inverse_transform(pred_scaled.reshape(-1, 1)).ravel()
    nn_preds.append(pred)
nn_preds = np.array(nn_preds)
nn_mean = np.mean(nn_preds, axis=0)
nn_disagreement = np.std(nn_preds, axis=0)
gp_mean, gp_std = gp.predict(f1_top_candidates, return_std=True)
for j, point in enumerate(f1_top_candidates):
    distances = cdist(point.reshape(1, -1), X).ravel()
    nearest_idx = np.argsort(distances)[:3]

## Local refinement

In [16]:
from scipy.spatial.distance import cdist
import numpy as np
rng = np.random.default_rng(42)
local_results = {}
for i in [4, 5, 7]:
    X = data[i].drop(columns='output').values.astype(float)
    y = data[i]['output'].values.astype(float)
    best_idx = np.argmax(y)
    best_point = X[best_idx]
    best_output = y[best_idx]
    n_candidates = 100000
    local_scale = 0.08
    candidates = best_point + rng.normal(0, local_scale, size=(n_candidates, X.shape[1]))
    candidates = np.clip(candidates, 0, 1)
    distances = cdist(candidates, X)
    nearest_distance = np.min(distances, axis=1)
    valid = nearest_distance > 1e-06
    candidates = candidates[valid]
    nearest_distance = nearest_distance[valid]
    gp_mean, gp_std = gp_models[i].predict(candidates, return_std=True)
    candidates_scaled = nn_x_scalers[i].transform(candidates)
    nn_predictions = []
    for model in nn_models[i]:
        pred_scaled = model.predict(candidates_scaled)
        pred = nn_y_scalers[i].inverse_transform(pred_scaled.reshape(-1, 1)).ravel()
        nn_predictions.append(pred)
    nn_predictions = np.array(nn_predictions)
    nn_mean = np.mean(nn_predictions, axis=0)
    nn_disagreement = np.std(nn_predictions, axis=0)
    gp_z = (gp_mean - np.mean(gp_mean)) / (np.std(gp_mean) + 1e-12)
    nn_z = (nn_mean - np.mean(nn_mean)) / (np.std(nn_mean) + 1e-12)
    combined_score = (gp_z + nn_z) / 2
    best_candidate_idx = np.argmax(combined_score)
    selected = candidates[best_candidate_idx]
    local_results[i] = selected

## Function 5 novelty constraint

In [17]:
i = 5
X = data[i].drop(columns='output').values.astype(float)
y = data[i]['output'].values.astype(float)
best_idx = np.argmax(y)
best_point = X[best_idx]
best_output = y[best_idx]
rng = np.random.default_rng(42)
n_candidates = 100000
local_scale = 0.08
candidates = best_point + rng.normal(0, local_scale, size=(n_candidates, X.shape[1]))
candidates = np.clip(candidates, 0, 1)
distances = cdist(candidates, X)
nearest_distance = np.min(distances, axis=1)
min_novelty = 0.03
valid = nearest_distance >= min_novelty
candidates = candidates[valid]
nearest_distance = nearest_distance[valid]
gp_mean, gp_std = gp_models[i].predict(candidates, return_std=True)
candidates_scaled = nn_x_scalers[i].transform(candidates)
nn_predictions = []
for model in nn_models[i]:
    pred_scaled = model.predict(candidates_scaled)
    pred = nn_y_scalers[i].inverse_transform(pred_scaled.reshape(-1, 1)).ravel()
    nn_predictions.append(pred)
nn_predictions = np.array(nn_predictions)
nn_mean = np.mean(nn_predictions, axis=0)
nn_disagreement = np.std(nn_predictions, axis=0)
gp_z = (gp_mean - np.mean(gp_mean)) / (np.std(gp_mean) + 1e-12)
nn_z = (nn_mean - np.mean(nn_mean)) / (np.std(nn_mean) + 1e-12)
combined_score = (gp_z + nn_z) / 2
best_candidate_idx = np.argmax(combined_score)
selected = candidates[best_candidate_idx]

## Final validation and submission

In [18]:
final_candidates = {1: np.array([0.998294, 0.000709]), 2: np.array([0.697103, 0.994806]), 3: np.array([0.463879, 0.497888, 0.452731]), 4: np.array([0.386248, 0.38101, 0.343907, 0.423092]), 5: np.array([1.0, 1.0, 0.97, 1.0]), 6: np.array([0.426973, 0.384648, 0.468006, 0.752822, 0.084308]), 7: np.array([0.157801, 0.305871, 0.468629, 0.230835, 0.339422, 0.619025]), 8: np.array([0.026226, 0.080912, 0.12825, 0.056836, 0.74825, 0.399531, 0.253527, 0.72075])}
all_valid = True
for i in range(1, 9):
    candidate = final_candidates[i]
    X = data[i].drop(columns='output').values.astype(float)
    correct_dimensions = len(candidate) == X.shape[1]
    within_bounds = np.all((candidate >= 0) & (candidate <= 1))
    distances = cdist(candidate.reshape(1, -1), X).ravel()
    nearest_distance = np.min(distances)
    duplicate = nearest_distance < 1e-06
    valid = correct_dimensions and within_bounds and (not duplicate)
    all_valid = all_valid and valid
for i in range(1, 9):
    formatted = ', '.join((f'{x:.6f}' for x in final_candidates[i]))

if not all_valid:
    raise ValueError("One or more final candidates failed validation")
for i in range(1, 9):
    print(f"F{i}: " + "-".join(f"{v:.6f}" for v in final_candidates[i]))

F1: 0.998294-0.000709
F2: 0.697103-0.994806
F3: 0.463879-0.497888-0.452731
F4: 0.386248-0.381010-0.343907-0.423092
F5: 1.000000-1.000000-0.970000-1.000000
F6: 0.426973-0.384648-0.468006-0.752822-0.084308
F7: 0.157801-0.305871-0.468629-0.230835-0.339422-0.619025
F8: 0.026226-0.080912-0.128250-0.056836-0.748250-0.399531-0.253527-0.720750
